In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

from multiprocessing import Pool
from scipy import stats
from scipy.stats import linregress
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from mpl_toolkits.basemap import Basemap
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter, LatitudeLocator
import matplotlib.colors as mcolors
import sys
import pyet
import os
import pandas as pd
from math import pi
from glob import glob
import xarray as xr
import numpy as np
from datetime import timedelta
import pickle
from climpred import metrics as cmet
from climpred.options import OPTIONS
from climpred import HindcastEnsemble
import climpred
import config.gefDataUtils as gData
import config.gefEToUtils as gET
import config.dataUtils as dUtils
import config.climpredUtils as clim
import config.gefPlotUtils as gPlot
import config.laggedAvgEnsemble as lagAvg
import config.unitTest as test
import config.HindcastMetrics as hindMetrics
import config.STATIC as call
import config.gefESRutils as gefESR
import config.FDtimeSeriesPlot as fdplot
import config.gefDetrend as gefTrend


'''Set this to ensure detrending is correct'''
climpred.set_options(seasonality="dayofyear") 
seasonality_str = OPTIONS['seasonality']


/glade/work/klesinger/conda-envs/tf212gpu_new/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
'''Data was previously download from AWS bucket using a different HPC system'''
vars_to_process=call.gefs_vars

# test.delete_bad_files_check_2(vars_to_process)
# test.find_bad_grib_to_netcdf_files_and_delete(var) #checks for bad files from grib to netcdf conversion

# Now we need to compute reference ETo (albedo = 0.23, standard bulk and surface resistance (see FAO-56 Penman Monteith methodology). Additionally we are only using downwelling shortwave radiation 

In [ ]:
'''Creating reference ET manually'''
# #Merge ensemble members into a single file (1043 initializations total)
# [gData.merge_ensemble_members(var) for var in vars_to_process] #single processing

#Multiprocessing
# if __name__ == '__main__':
#     p=Pool(len(vars_to_process))
#     p.map(gData.merge_ensemble_members,vars_to_process)



In [ ]:
# gData.merge_CPC_reference_ET_files()

In [ ]:
tempName='spfh_2m'
fileDir=f'{call.gefs_dir}/GEFSv12_merged'
initDates= gData.get_init_date_list(path=f'{fileDir}/{tempName}/*.nc')

# [gET.create_FAO56_refET_GEFSv12(_date) for _date in initDates] #single-processing

# p=Pool(50)
# p.map(gET.create_FAO56_refET_GEFSv12,initDates) #multi-processing


In [ ]:

land_mask_vals = dUtils.load_CONUS_mask()['CONUS_mask'][0,:,:].values #Values of 1 indicate a land area

if recompute:
    if cpc_analysis == False:
        save_dir = f'{call.gefs_dir}/ESR_hindcast'
    else:
        save_dir = f'{call.gefs_dir}/ESR_hindcast_cpc_source'
        
    os.makedirs(save_dir, exist_ok=True)
    print('Recomputing calculation of ESR on GEFSv12 hindcast data')
    for _date in initDates:
        save_file = f'{save_dir}/esr_{_date}.nc'

        # break
        print(f'Making ESR for date {_date}.')
        EVP = xr.open_dataset(f'{call.gefs_dir}/GEFSv12_merged/lhtfl_sfc/lhtfl_sfc_{_date}.nc')
        EVP.close()
        if cpc_analysis == False:
            REFET = xr.open_dataset(f'{call.gefs_dir}/ETo_hindcast/refet_{_date}.nc')
        else:
            REFET = xr.open_dataset(f'{call.gefs_dir}/ETo_hindcast_cpc_source/merged_inits/refet_julian_{_date}.nc')
            REFET['L'] = np.arange(len(REFET.L.values))
            
        REFET.close()
        REFET=REFET.rename({'ETo_Penman':'data'}) #Do this so that it matches the same as observations
        esr = gefsv12_ESR_mask_and_interpolate_and_running_mean(EVP,REFET,land_mask_vals,num_days_in_rolling_mean=call.mean_rolling_length)

# First we must check how well refet matches with NLDAS refet 
## The code can eventually plot pearson r (and eventually ACC). The pearson r code is there, but I didn't make a figure script yet

In [ ]:
hindMetrics.compute_CRPS_on_hindcast_data(num_days_in_rolling_mean=call.mean_rolling_length, varname = 'cpc_refet')
hindMetrics.compute_CRPS_on_hindcast_data(num_days_in_rolling_mean=call.mean_rolling_length, varname = 'et')
hindMetrics.compute_CRPS_on_hindcast_data(num_days_in_rolling_mean=call.mean_rolling_length, varname = 'refet')

# Lagged average ensemble to ensure that our data looks reasonable

In [ ]:
[lagAvg.lagged_average_ensemble_mean(variable, recompute=False) for variable in ['et','refet','cpc_refet']]

'''Then plot the accumulated data over the year (then averaged over the period)'''
gPlot.GEF_yearly_average_statistics_of_accumulated_value()

# Now create Evaporative Stress Ratio with the same parameters as the observations

In [ ]:
gefESR.create_ESR_GEFSv12_hindcast(initDates = initDates,recompute=True, cpc_analysis = True)

In [ ]:
hindMetrics.compute_CRPS_on_hindcast_data(num_days_in_rolling_mean=call.mean_rolling_length, varname = 'ESR_refet')

In [ ]:
def GEF_monthly_average_statistics_of_accumulated_value(obs: xr.DataArray, year_ranges_tuple_1, year_ranges_tuple_2):
    cmap = plt.get_cmap('YlOrBr')    
    
    save_dir = f'Figures/et_pet_refet_monthly_and_yearly_sum_statistics'
    os.system(f'mkdir -p {save_dir}')

    land_mask = dutils.load_CONUS_mask()['CONUS_mask'][0,:,:].values
    
    monthly_sum_per_year = obs.groupby('time.year').apply(lambda x: x.groupby('time.month').sum(dim='time'))
    # Define custom month order starting from November (month 11)
    # custom_month_order = [12, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
    
    # Reindex the monthly sums to match the custom month order
    # monthly_sum_per_year = monthly_sum_per_year.reindex(month=custom_month_order)
    
    yr_plot1 = monthly_sum_per_year.sel(year=slice(str(year_ranges_tuple_1[0]),str(year_ranges_tuple_1[1]))).mean(dim='year')
    yr_plot2 = monthly_sum_per_year.sel(year=slice(str(year_ranges_tuple_2[0]),str(year_ranges_tuple_2[1]))).mean(dim='year')

    data_arrays1 = [yr_plot1['EVP'],yr_plot1['PEVPR'],yr_plot1['refET']]
    data_arrays2 = [yr_plot2['EVP'],yr_plot2['PEVPR'],yr_plot2['refET']]
   
    lon = yr_plot1.lon.values
    lat = yr_plot1.lat.values

    for idx1,data_arrays in enumerate([data_arrays1,data_arrays2]):

        for idx2, (arr, var_name) in enumerate(zip(data_arrays,['EVP','PET','refET',])):
            fig, axs = plt.subplots(
                nrows = 4, ncols= 3, subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(15, 15),dpi=300)
                
            axs = axs.flatten()
            
            axs_start = 0
            for idx3,month in enumerate(arr.month.values):
                month_name = calendar.month_name[month]
                # break
                data = arr.sel(month=month).values
                data = np.where(np.isnan(land_mask),np.nan,data)
                # data = np.ma.masked_invalid(data)
                for Y in range(data.shape[0]):
                    for X in range(data.shape[1]):
                        if np.isnan(land_mask[Y,X]) or (data[Y,X] == 0):
                            data[Y,X]=np.nan
                            
                v = np.linspace(np.nanmin(data), np.nanmax(data), 8, endpoint=True)
                v=np.array([round(i) for i in v])
                map = Basemap(projection='cyl', llcrnrlat=25, urcrnrlat=50,
                      llcrnrlon=-128, urcrnrlon=-60, resolution='l')
                x, y = map(*np.meshgrid(lon, lat))
        
                im = axs[axs_start].contourf(x, y, data, levels=v, extend='both',
                              transform=ccrs.PlateCarree(), cmap=cmap)
                gl = axs[axs_start].gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                                           linewidth=0.7, color='gray', alpha=0.5, linestyle='--')
                gl.top_labels = False
                gl.right_labels = False
                gl.left_labels = True
                gl.bottom_labels = True
                gl.xformatter = LongitudeFormatter()
                gl.yformatter = LatitudeFormatter()
                axs[axs_start].coastlines()
                axs[axs_start].set_aspect('equal')  # this makes the plots better

                if idx1==0:
                    axs[axs_start].set_title(f'{month_name} (1981-2020)',fontsize=12)
                else:
                    axs[axs_start].set_title(f'{month_name} (2000-2019)',fontsize=12)
                    
                cbar = fig.colorbar(im, ax = axs[axs_start], orientation='horizontal')
                cbar.set_label('mm/year', fontsize=10, labelpad=5)
                axs_start+=1
    
            plt.suptitle(f'Accumulated monthly accumulated {var_name}.\nThen averaged over the period. ',fontsize=20)
            if idx1 == 0:
                plt.savefig(f'{save_dir}/monthly_{var_name}_sum_{year_ranges_tuple_1[0]}-{year_ranges_tuple_1[1]}.png')
            else:
                plt.savefig(f'{save_dir}/monthly_{var_name}_sum_{year_ranges_tuple_2[0]}-{year_ranges_tuple_2[1]}.png')

    return('Completed monthly sum plots')
    return(0)


# Detrend ESR by day of year 
## Source: https://climpred.readthedocs.io/en/stable/prediction-ensemble-object.html?highlight=detrend#detrend

In [ ]:

if cpc_source == True:
    esr_list = sorted(glob(f'{call.gefs_dir}/ESR_hindcast_cpc_source/*.nc'))
    save_dir = f'{call.gefs_dir}/ESR_hindcast_detrended_cpc_source'
else:
    esr_list = sorted(glob(f'{call.gefs_dir}/ESR_hindcast/*.nc'))
    save_dir = f'{call.gefs_dir}/ESR_hindcast_detrended'

os.makedirs(save_dir, exist_ok=True)

# first open up all the files, this will make it easier later when performing the standardization
#Only take the ensemble mean for the hindcast assessment
datasets_int = [xr.open_dataset(f).mean(dim='M',skipna=True) for f in esr_list]
for idx,file in enumerate(datasets_int):
    new_date = datasets_int[idx]
    new_date['S'] = np.atleast_1d(pd.to_datetime(new_date.S.values))
    datasets_int[idx] = new_date

# datasets_int = gefTrend.convert_int_leads_to_day_of_year(datasets_int) #do not do this, it creates a very large file
gef = xr.concat(datasets_int, dim='S')

coeff = gef.polyfit(dim='S', deg=1)

if isinstance(gef, xr.Dataset):
    for v in coefficients:
        name = v.replace("_polyfit_coefficients", "")
        fit = xr.polyval(coord, coefficients[v]).rename(name)
        fits.append(fit)
    fits = xr.merge(fits)

with xr.set_options(keep_attrs=True):
    ds_rm_poly = gef - fits

final_OUT = (ds_rm_poly - ds_rm_poly.min(dim=['S'])) / (ds_rm_poly.max(dim=['S']) -ds_rm_poly.min(dim=['S']))
final_OUT.to_netcdf(f'{save_dir}/ESR_detrended_cpc_source_full.nc')


In [ ]:
### to check if the output is any good -- currenlty not
final_OUT.sel(S='2012-07-21',method='nearest').data[25,:,:].plot()

In [ ]:
'''

old code that almost works, but it still doesn't look right



Now that we have the slope and intercept saved, we can open the files individually, take the mean, then perform equation
x - (slope * t + intercept)

gef_trnd_clim,land_mask = gefTrend.save_ESR_detrend_vals_day_of_year_climatology(climatology=call.short_clim, cpc_source = True)



Now for each file, we go through and subtract based on the index (as t).
dtrnd_OUT = gef.copy(deep=True)
year_start = pd.to_datetime(gef.S.values[0]).year

t_val = 0
for iDate, date in enumerate(gef.S.values):
    # iDate, date = 0, np.datetime64('2000-01-05T00:00:00.000000000')
    if (pd.to_datetime(gef.S.values[iDate]).year != pd.to_datetime(gef.S.values[iDate-1]).year) and (iDate !=0):
        t_val +=1
    julian_dates = gData.julian_date_HINDCAST(gef.S.values[0],gef.L.shape[0])
    gef['L'] = julian_dates
    dtrnd_OUT.data[:,:,:,:] = gef.data.values - (gef_trnd_clim.slope.sel(L=julian_dates).values*t_val + gef_trnd_clim.intercept.sel(L=julian_dates).values)


    
then we perform min max standardization

dtrnd_OUT = dtrnd_OUT.clip(min=-20, max=20)
max_ = dtrnd_OUT.data.max(dim=['S','L']).values
min_
#Now resave the entire data as a single file
'''


In [ ]:
###### For detrending ESR_pet and ESR_refet######
# Define a detrending function
def detrend_doy(x):
    """Remove the linear trend from data."""
    t = np.arange(len(x))
    mask = np.isfinite(x)
    if np.sum(mask) > 1:  # Ensure there are enough points to fit a line
        slope, intercept, r_value, p_value, std_err = stats.linregress(t[mask], x[mask])
        out_val = x - (slope * t + intercept)
        return out_val
    else:
        return x  # Not enough points to detrend, return original

In [ ]:
save_ESR_detrend_vals_day_of_year_climatology(climatology=call.short_clim, cpc_source = True)
